In [1]:
from nbiatoolkit.models.nbia_responses import (
  Patient,
  PatientList,
  Study,
  StudyList,
  Series,
  SeriesList,
)
from nbiatoolkit import NBIA_BASE_URLS, NBIA_ENDPOINTS, NBIAClient
import requests
from dataclasses import dataclass, field, asdict
from pandas import DataFrame
from typing import Optional, List, Dict, Any, Union, Tuple
from typing import Optional
from pydantic import BaseModel, Field
from enum import Enum

import asyncio
import aiohttp
import aiofiles
from datetime import datetime
import pathlib
import json
import time 


In [2]:
import structlog 
logger = structlog.get_logger()
logger.debug("Starting NBIA Client")
logger.info("Starting NBIA Client")

2024-12-18 14:04:54 [debug    ] Starting NBIA Client          
2024-12-18 14:04:54 [info     ] Starting NBIA Client          


In [3]:
client = NBIAClient()
collections = client.getCollections(return_type="dataframe")
clist = collections.Collection.to_list()


In [26]:
async def fetch_and_write(
	session: aiohttp.ClientSession,
	url: str,
	params: Dict[str, str],
	headers: Optional[Dict[str, str]],
	file_name: pathlib.Path,
	semaphore: asyncio.Semaphore,
	**kwargs: Any
) -> None:
	"""
	Fetches data from the given URL and writes it to a file asynchronously, respecting semaphore limits.

	Parameters
	----------
	session : aiohttp.ClientSession
		An aiohttp session to use for making requests.
	url : str
		The API endpoint URL to fetch data from.
	params : dict
		Parameters to include in the GET request.
	headers : dict, optional
		Headers to include in the GET request.
	file_name : pathlib.Path
		The Path object representing the file to write the content to.
	semaphore : asyncio.Semaphore
		A semaphore to limit the number of concurrent tasks.
	kwargs : Any
		Additional keyword arguments for logging context.

	Returns
	-------
	None
	"""
	async with semaphore:
		thislogger = logger.bind(url=url, **kwargs)
		thislogger.debug("Start fetching")

		async with session.get(url, params=params, headers=headers) as response:
			# Ensure the response is JSON
			if response.headers.get('Content-Type') == 'application/json':
				content = await response.json()
				thislogger.debug("Received JSON response")
			else:
				thislogger.warning("Unexpected Content-Type: %s", response.headers.get('Content-Type'))
				content = None

	# # Write the JSON content to the file
	# if content is not None:
	# 	async with aiofiles.open(file_name, mode='w') as file:
	# 		await file.write(json.dumps(content, indent=4))  # Pretty print JSON
	# 		thislogger.debug("Written JSON response to file")

async def main(
	query_tuples: List[Tuple[str, Dict[str, str], pathlib.Path]],
	conc_requests: int = 5,
	headers: Optional[Dict[str, str]] = None
) -> None:
	"""
	Main function to fetch data from an API concurrently and write responses to files.
	Uses a semaphore to limit concurrency.

	Parameters
	----------
	query_tuples : list of tuple
		A list of tuples containing URL, request parameters, and output file paths.
	conc_requests : int, optional
		The maximum number of concurrent requests, by default 5.

	Returns
	-------
	None
	"""
	tasks = []
	semaphore = asyncio.Semaphore(conc_requests)  # Limit concurrency

	async with aiohttp.ClientSession() as session:
		for query_tuple in query_tuples:
			url, params, file_name = query_tuple
			task = fetch_and_write(
				session,
				url,
				params,
				headers,
				file_name,
				semaphore,
				collection=params["Collection"]
			)
			tasks.append(task)
		await asyncio.gather(*tasks)

# semaphore = asyncio.Semaphore(10)  # Limit concurrency

# async with aiohttp.ClientSession() as session:
# 	await fetch_and_write(
# 		session,
# 		NBIA_BASE_URLS.NBIA.value + NBIA_ENDPOINTS.GET_SERIES.value,
# 		{},
# 		client.headers, 
# 		pathlib.Path("data") / "metadata" / "all_series.json",
# 		semaphore
# 	)


In [23]:
data_path = pathlib.Path("data")

In [28]:

# Check if there's an existing event loop and run the main function accordingly
try:
	loop = asyncio.get_running_loop()
except RuntimeError:
	loop = None

# URL
CONCURRENT_REQUESTS = 25

# api_url = NBIA_BASE_URLS.NBIA.value + NBIA_ENDPOINTS.GET_PATIENTS.value  # Example API endpoint
endpoints = {
	# "Patient": NBIA_ENDPOINTS.GET_PATIENTS.value,
	# "Studies": NBIA_ENDPOINTS.GET_STUDIES.value,
	"Series" : NBIA_ENDPOINTS.GET_SERIES.value
}

# Tuples of URL, parameters, and file names
query_tuples = []
for heading, endpoint in endpoints.items():
	url = NBIA_BASE_URLS.NBIA.value + endpoint
	for collection in clist[:25]:
		query_tuples.append(
			(
				url, 
				{"Collection": collection }, 
				data_path / "metadata" / collection / f"{heading}List_{collection}.json")
			)

# make all the directories
for _, _, file_name in query_tuples:
	file_name.parent.mkdir(parents=True, exist_ok=True)

start = time.time()
if loop and loop.is_running():
	# If there's an existing event loop, create a task for the main function
	task = asyncio.create_task(main(query_tuples=query_tuples, conc_requests=CONCURRENT_REQUESTS, headers = client.headers))
	await task  # Ensure the task completes before logging the time taken
else:
	# If there's no existing event loop, run the main function
	asyncio.run(main(query_tuples=query_tuples[:10], conc_requests=CONCURRENT_REQUESTS, headers = client.headers))

logger.info("Time taken", time_taken = time.time() - start)

2024-12-18 14:30:42 [debug    ] Start fetching                 collection=4D-Lung url=https://services.cancerimagingarchive.net/nbia-api/services/v2/getSeries
2024-12-18 14:30:42 [debug    ] Start fetching                 collection=ACRIN-6698 url=https://services.cancerimagingarchive.net/nbia-api/services/v2/getSeries
2024-12-18 14:30:42 [debug    ] Start fetching                 collection=ACRIN-Contralateral-Breast-MR url=https://services.cancerimagingarchive.net/nbia-api/services/v2/getSeries
2024-12-18 14:30:42 [debug    ] Start fetching                 collection=ACRIN-FLT-Breast url=https://services.cancerimagingarchive.net/nbia-api/services/v2/getSeries
2024-12-18 14:30:42 [debug    ] Start fetching                 collection=ACRIN-NSCLC-FDG-PET url=https://services.cancerimagingarchive.net/nbia-api/services/v2/getSeries
2024-12-18 14:30:42 [debug    ] Start fetching                 collection=Adrenal-ACC-Ki67-Seg url=https://services.cancerimagingarchive.net/nbia-api/services/

In [12]:
# patient_files = list(data_path.glob("**/PatientList*.json"))

# for i, file in enumerate(patient_files):
#   collection = file.parent.name
#   with file.open(mode="r") as f:
#     data = json.loads(f.read())
#     patient_list = PatientList(items=Patient.from_dicts(data))
collection_paths = list(data_path.glob("metadata2/*"))
print(len(collection_paths))
for collection_path in collection_paths:
  collection = collection_path.name
  if collection != "4D-Lung":
    continue
  series_file = list(collection_path.glob("SeriesList*.json"))[0]
  with series_file.open(mode="r") as f:
    series_list = json.loads(f.read())
  print(len(series_list))
  break

128
6690


In [16]:
_sel = Series.from_dicts(series_list)
sl = SeriesList(items=_sel)

In [30]:
sl.df

,Collection,SeriesInstanceUID,StudyInstanceUID,Modality,ProtocolName,SeriesDate,SeriesDescription,BodyPartExamined,SeriesNumber,AnnotationsFlag,...,ImageCount,TimeStamp,LicenseName,LicenseURI,CollectionURI,FileSize,DateReleased,StudyDescription,StudyDate,ThirdPartyAnalysis
0,4D-Lung,1.3.6.1.4.1.14519.5.2.1.6834.5010.189721824525...,1.3.6.1.4.1.14519.5.2.1.6834.5010.552215730027...,CT,5.1 4DCT & ITV FB + 4D + INSP/EXP,1997-10-03,"P4^P100^S113^I0, Gated, 70.0%",LUNG,507,None,...,50,2015-07-20 17:58:54.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGLE,26405988,2015-07-20 17:58:54,None,1997-10-03,None
1,4D-Lung,1.3.6.1.4.1.14519.5.2.1.6834.5010.336250251691...,1.3.6.1.4.1.14519.5.2.1.6834.5010.980344486630...,CT,5.1 4DCT & ITV FB + 4D + INSP/EXP,1997-10-07,"P4^P100^S116^I0, Gated, 70.0%",LUNG,507,None,...,50,2015-07-20 17:40:07.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGLE,26405988,2015-07-20 17:40:07,None,1997-10-07,None
2,4D-Lung,1.3.6.1.4.1.14519.5.2.1.6834.5010.227929163446...,1.3.6.1.4.1.14519.5.2.1.6834.5010.157653211810...,CT,5.1 4DCT & ITV FB + 4D + INSP/EXP,1997-09-18,"P4^P100^S104^I0, Gated, 30.0%",LUNG,503,None,...,50,2015-07-20 17:56:27.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGLE,26405988,2015-07-20 17:56:27,None,1997-09-18,None
3,4D-Lung,1.3.6.1.4.1.14519.5.2.1.6834.5010.925990093742...,1.3.6.1.4.1.14519.5.2.1.6834.5010.256783235670...,CT,5.1 4DCT & ITV FB + 4D + INSP/EXP,1997-10-14,"P4^P100^S123^I0, Gated, 20.0%",LUNG,502,None,...,50,2015-07-20 17:55:30.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGLE,26405988,2015-07-20 17:55:30,None,1997-10-14,None
4,4D-Lung,1.3.6.1.4.1.14519.5.2.1.6834.5010.139116724721...,1.3.6.1.4.1.14519.5.2.1.6834.5010.980344486630...,CT,5.1 4DCT & ITV FB + 4D + INSP/EXP,1997-10-07,"P4^P100^S116^I0, Gated, 10.0%",LUNG,501,None,...,50,2015-07-20 17:51:12.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGLE,26405988,2015-07-20 17:51:12,None,1997-10-07,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6685,4D-Lung,2.25.202457580701300636488706306139237962947.75,1.3.6.1.4.1.14519.5.2.1.6834.5010.197990761062...,RTSTRUCT,None,NaT,"P4^P119^S307^I00007, Gated, 40.0%A",LUNG,1,None,...,1,2016-10-12 15:18:23.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGLE,81604,2016-10-12 15:18:23,None,2001-01-18,None
6686,4D-Lung,2.25.202457580701300636488706306139237962947.64,1.3.6.1.4.1.14519.5.2.1.6834.5010.172668278160...,RTSTRUCT,None,NaT,"P4^P119^S306^I00006, Gated, 30.0%A",LUNG,1,None,...,1,2016-10-12 15:18:43.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGLE,97388,2016-10-12 15:18:43,None,2001-01-14,None
6687,4D-Lung,2.25.202457580701300636488706306139237962947.41,1.3.6.1.4.1.14519.5.2.1.6834.5010.193736285012...,RTSTRUCT,None,NaT,"P4^P119^S304^I00003, Gated, 0.0%A",LUNG,1,None,...,1,2016-10-12 15:18:50.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGLE,114092,2016-10-12 15:18:50,None,2001-01-08,None
6688,4D-Lung,2.25.202457580701300636488706306139237962947.15,1.3.6.1.4.1.14519.5.2.1.6834.5010.115505120169...,RTSTRUCT,None,NaT,"P4^P119^S301^I00007, Gated, 40.0%A",LUNG,1,None,...,1,2016-10-12 15:18:33.0,Creative Commons Attribution 3.0 Unported License,http://creativecommons.org/licenses/by/3.0/,https://doi.org/10.7937/K9/TCIA.2016.ELN8YGL

In [31]:
sl.filter(Modality == "CT")

NameError: name 'Modality' is not defined